# FR1 and FR5 algorithm prototype

This notebook prototypes PNG input and image LSB replacement without a GUI. It uses the real `payload_protocol.py` packet so that framing, hashing, and extraction assumptions are tested early.

Prototype decisions:

- Support `RGB` and `RGBA` PNG files. Embed only in RGB channels; preserve alpha.
- Number channels in row-major pixel order: R, G, B. A start location is an index in that channel sequence.
- Use the protocol packet's existing four-byte payload length; do not add a second FR5 header.
- Hash a canonical pixel representation with the selected RGB LSBs cleared. This representation is stable before and after embedding.
- Treat key selection, start-location derivation, and replay detection as external responsibilities.


In [ ]:
from io import BytesIO
from pathlib import Path
import struct
import sys

from PIL import Image, UnidentifiedImageError

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from payload_protocol import (
    create_payload,
    generate_rsa_keypair,
    pack_verification_packet,
    unpack_and_verify_packet,
)


## FR1: PNG loading and validation

`load_png` fully loads and copies the image before the source file closes. Unsupported formats and modes fail explicitly instead of being converted silently.


In [ ]:
SUPPORTED_MODES = {"RGB", "RGBA"}


def load_png(source) -> Image.Image:
    try:
        with Image.open(source) as opened:
            if opened.format != "PNG":
                raise ValueError(f"Expected PNG input, received {opened.format or 'unknown'}")
            opened.load()
            if opened.mode not in SUPPORTED_MODES:
                raise ValueError(
                    f"Unsupported PNG mode {opened.mode!r}; expected RGB or RGBA"
                )
            return opened.copy()
    except (OSError, UnidentifiedImageError) as exc:
        raise ValueError("Unreadable or corrupted image input") from exc


def image_metadata(image: Image.Image) -> dict:
    if image.mode not in SUPPORTED_MODES:
        raise ValueError(f"Unsupported image mode {image.mode!r}")
    return {
        "width": image.width,
        "height": image.height,
        "mode": image.mode,
        "embeddable_channels": image.width * image.height * 3,
    }


## FR5: canonical hashing, capacity, and LSB replacement

Raw PNG bytes cannot be hashed for verification because embedding changes them. `canonical_cover_bytes` clears the selected RGB LSBs and includes the image dimensions, mode, normalized RGB values, and unchanged alpha values. The same representation can therefore be computed from the original and stego images.


In [ ]:
CANONICAL_FORMAT = b"FR1FR5-COVER-v1\x00"


def _validate_image(image: Image.Image) -> None:
    if image.mode not in SUPPORTED_MODES:
        raise ValueError(f"Unsupported image mode {image.mode!r}")


def _validate_lsb_count(lsb_count: int) -> None:
    if isinstance(lsb_count, bool) or not isinstance(lsb_count, int):
        raise TypeError("lsb_count must be an integer")
    if not 1 <= lsb_count <= 8:
        raise ValueError("lsb_count must be from 1 through 8")


def _channel_count(image: Image.Image) -> int:
    return image.width * image.height * 3


def _validate_start(image: Image.Image, start_channel: int) -> None:
    if isinstance(start_channel, bool) or not isinstance(start_channel, int):
        raise TypeError("start_channel must be an integer")
    if not 0 <= start_channel <= _channel_count(image):
        raise ValueError("start_channel is outside the RGB channel sequence")


def canonical_cover_bytes(image: Image.Image, lsb_count: int) -> bytes:
    _validate_image(image)
    _validate_lsb_count(lsb_count)
    clear_mask = (0xFF << lsb_count) & 0xFF
    mode_bytes = image.mode.encode("ascii")
    result = bytearray(CANONICAL_FORMAT)
    result.extend(struct.pack(">IIB", image.width, image.height, len(mode_bytes)))
    result.extend(mode_bytes)

    pixels = image.load()
    for y in range(image.height):
        for x in range(image.width):
            pixel = pixels[x, y]
            result.extend(channel & clear_mask for channel in pixel[:3])
            if image.mode == "RGBA":
                result.append(pixel[3])
    return bytes(result)


def image_capacity_bytes(
    image: Image.Image, lsb_count: int, start_channel: int
) -> int:
    _validate_image(image)
    _validate_lsb_count(lsb_count)
    _validate_start(image, start_channel)
    available_channels = _channel_count(image) - start_channel
    return (available_channels * lsb_count) // 8


def _symbol_from_bytes(data: bytes, bit_offset: int, width: int) -> int:
    value = 0
    for relative_bit in range(width):
        value <<= 1
        absolute_bit = bit_offset + relative_bit
        if absolute_bit < len(data) * 8:
            byte = data[absolute_bit // 8]
            value |= (byte >> (7 - absolute_bit % 8)) & 1
    return value


def embed_lsb(
    image: Image.Image, data: bytes, lsb_count: int, start_channel: int
) -> Image.Image:
    _validate_image(image)
    _validate_lsb_count(lsb_count)
    _validate_start(image, start_channel)
    if not isinstance(data, bytes):
        raise TypeError("data must be bytes")

    required_channels = (len(data) * 8 + lsb_count - 1) // lsb_count
    if required_channels > _channel_count(image) - start_channel:
        raise ValueError(
            f"Payload requires {required_channels} channels, but only "
            f"{_channel_count(image) - start_channel} are available"
        )

    stego = image.copy()
    pixels = stego.load()
    replace_mask = (1 << lsb_count) - 1
    clear_mask = 0xFF ^ replace_mask

    for offset in range(required_channels):
        channel_index = start_channel + offset
        pixel_index, channel = divmod(channel_index, 3)
        x = pixel_index % stego.width
        y = pixel_index // stego.width
        pixel = list(pixels[x, y])
        symbol = _symbol_from_bytes(data, offset * lsb_count, lsb_count)
        pixel[channel] = (pixel[channel] & clear_mask) | symbol
        pixels[x, y] = tuple(pixel)

    return stego


## Extraction compatibility helper

This is not the final FR8 implementation. It proves the FR5 bit order and packet framing. It first reads the protocol's four-byte JSON length, derives the RSA signature size from the public key, and then reads the complete packet.


In [ ]:
def extract_bytes(
    image: Image.Image, byte_count: int, lsb_count: int, start_channel: int
) -> bytes:
    _validate_image(image)
    _validate_lsb_count(lsb_count)
    _validate_start(image, start_channel)
    if byte_count < 0:
        raise ValueError("byte_count cannot be negative")

    required_channels = (byte_count * 8 + lsb_count - 1) // lsb_count
    if required_channels > _channel_count(image) - start_channel:
        raise ValueError("Requested bytes exceed the available image capacity")

    pixels = image.load()
    symbol_mask = (1 << lsb_count) - 1
    accumulator = 0
    accumulated_bits = 0
    output = bytearray()

    for offset in range(required_channels):
        channel_index = start_channel + offset
        pixel_index, channel = divmod(channel_index, 3)
        x = pixel_index % image.width
        y = pixel_index // image.width
        accumulator = (accumulator << lsb_count) | (pixels[x, y][channel] & symbol_mask)
        accumulated_bits += lsb_count

        while accumulated_bits >= 8 and len(output) < byte_count:
            accumulated_bits -= 8
            output.append((accumulator >> accumulated_bits) & 0xFF)
            accumulator &= (1 << accumulated_bits) - 1

    return bytes(output)


def extract_protocol_packet(
    image: Image.Image, public_key, lsb_count: int, start_channel: int
) -> bytes:
    length_header = extract_bytes(image, 4, lsb_count, start_channel)
    payload_length = struct.unpack(">I", length_header)[0]
    signature_length = (public_key.key_size + 7) // 8
    packet_length = 4 + payload_length + signature_length
    return extract_bytes(image, packet_length, lsb_count, start_channel)


## End-to-end prototype

This cell creates an image in memory, builds and embeds a real signed packet, saves and reloads the PNG, extracts the packet, and verifies it against the canonical stego representation. It also changes a non-LSB image bit to demonstrate a `Tampered` verdict without damaging the embedded packet.


In [ ]:
cover = Image.new("RGB", (64, 64))
cover_pixels = cover.load()
for y in range(cover.height):
    for x in range(cover.width):
        cover_pixels[x, y] = (
            (3 * x + 5 * y) % 256,
            (7 * x + 11 * y) % 256,
            (13 * x + 17 * y) % 256,
        )

lsb_count = 2
start_channel = 17
original_pixels = cover.tobytes()
private_key, public_key = generate_rsa_keypair()
stable_cover = canonical_cover_bytes(cover, lsb_count)
payload = create_payload(
    "prototype-image",
    stable_cover,
    {"format": "PNG", "lsb_count": lsb_count},
)
packet = pack_verification_packet(payload, private_key)
assert len(packet) <= image_capacity_bytes(cover, lsb_count, start_channel)

stego = embed_lsb(cover, packet, lsb_count, start_channel)
assert cover.tobytes() == original_pixels
assert canonical_cover_bytes(stego, lsb_count) == stable_cover

png_buffer = BytesIO()
stego.save(png_buffer, format="PNG")
png_buffer.seek(0)
reloaded_stego = load_png(png_buffer)
extracted_packet = extract_protocol_packet(
    reloaded_stego, public_key, lsb_count, start_channel
)
verification = unpack_and_verify_packet(
    extracted_packet, public_key, canonical_cover_bytes(reloaded_stego, lsb_count)
)
assert verification[0:2] == (True, "Authentic")

tampered = reloaded_stego.copy()
tampered_pixels = tampered.load()
last_pixel = list(tampered_pixels[tampered.width - 1, tampered.height - 1])
last_pixel[0] ^= 0x80
tampered_pixels[tampered.width - 1, tampered.height - 1] = tuple(last_pixel)
tampered_verification = unpack_and_verify_packet(
    extracted_packet, public_key, canonical_cover_bytes(tampered, lsb_count)
)
assert tampered_verification[0:2] == (False, "Tampered")

{
    "metadata": image_metadata(cover),
    "capacity_bytes": image_capacity_bytes(cover, lsb_count, start_channel),
    "packet_bytes": len(packet),
    "verification": verification[:2],
    "tampered_verification": tampered_verification[:2],
}


## Integration limits exposed by the prototype

- The verifier must know or derive the same LSB count and start channel before extraction.
- The packet has no magic value, protocol version, algorithm identifier, or key identifier. A wrong start location is likely to appear as corrupt or oversized framing.
- The canonical hash intentionally ignores the selected low bits in all RGB channels. Changes confined to those bits outside the signed packet may not be detected. At 8 LSBs, no RGB cover content remains available to the canonical hash.
- PNG ancillary metadata is not authenticated by the canonical representation.
- The nonce does not prevent replay unless another component records and rejects previously used values.
- Public-key distribution and trust remain outside this notebook. Private keys must not be committed.
